# 3 тапсырма – Ықтималдық, Байестік қорытындылау және HMM
**Студент:** Сұлтанов Айбар, 1-топ

## Бастапқы деректер

In [ ]:
# ---------- Бастапқы деректер ----------
P_H = 0.20                      # P(HAZARD)
P_ALARM = {"H": 0.90, "S": 0.10}   # P(ALARM | күй)
P_NORMAL = {"H": 0.10, "S": 0.90}  # P(NORMAL | күй)

# Ауысу ықтималдықтары
P_HH = 0.80                     # P(H_t | H_{t-1})
P_SS = 0.90                     # P(S_t | S_{t-1})
P_HS = 1 - P_SS                 # P(H_t | S_{t-1}) = 0.10

observations = ["ALARM", "NORMAL", "ALARM", "ALARM"]


def likelihood(obs, state):
    """P(бақылау | күй)"""
    return P_ALARM[state] if obs == "ALARM" else P_NORMAL[state]

## A. Байес формуласы

In [ ]:
# ---------- A. Байес формуласы ----------
def bayes(prior_h, obs):
    """P(H | E) = P(E|H)·P(H) / P(E)"""
    p_e_h = likelihood(obs, "H")
    p_e_s = likelihood(obs, "S")
    p_e = p_e_h * prior_h + p_e_s * (1 - prior_h)   # толық ықтималдық
    posterior = p_e_h * prior_h / p_e
    return p_e, posterior


print("A. Байес формуласы")
for obs in ["ALARM", "NORMAL"]:
    p_e, post = bayes(P_H, obs)
    print(f"  {obs:6}: P(H)={P_H:.2f}, P(E|H)={likelihood(obs, 'H'):.2f}, "
          f"P(E)={p_e:.2f}, P(H|E)={post:.4f}")

## D. Сезімталдық талдауы

In [ ]:
# ---------- D. Сезімталдық талдауы ----------
print("\nD. Сезімталдық эксперименті (бақылау: ALARM)")
for i, prior in enumerate([0.05, 0.20, 0.50, 0.80], start=1):
    _, post = bayes(prior, "ALARM")
    print(f"  Эксперимент {i}: бастапқы P(H)={prior:.2f} -> соңғы P(H)={post:.4f}")

## C. HMM сүзгісі

In [ ]:
# ---------- C. HMM сүзгісі ----------
def hmm_filter(prior_h, obs_seq):
    belief_h = prior_h
    results = []
    for t, obs in enumerate(obs_seq, start=1):
        # 1) Болжау (prediction)
        pred_h = P_HH * belief_h + P_HS * (1 - belief_h)
        pred_s = 1 - pred_h
        # 2) Бақылау негізінде жаңарту
        un_h = likelihood(obs, "H") * pred_h
        un_s = likelihood(obs, "S") * pred_s
        # 3) Нормалау
        belief_h = un_h / (un_h + un_s)
        results.append((t, obs, pred_h, belief_h, pred_s, 1 - belief_h))
    return results


print("\nC. HMM сүзгілеу")
print(f"  {'t':<3}{'Бақылау':<9}{'Болж.P(H)':>10}{'Жаң.P(H)':>10}"
      f"{'Болж.P(S)':>11}{'Жаң.P(S)':>10}")
for t, obs, ph, uh, ps, us in hmm_filter(P_H, observations):
    print(f"  t{t:<2}{obs:<9}{ph:>10.4f}{uh:>10.4f}{ps:>11.4f}{us:>10.4f}")